In [1]:
import numpy as np
import pandas as pd

In [2]:
NUM_SERVICE_VISITS = 5000

In [3]:
job_card_ids = [
    f"JC{i:06d}"
    for i in range(1, NUM_SERVICE_VISITS + 1)
]

In [4]:
NUM_CUSTOMERS = 1800

In [5]:
customer_pool = [
    f"C{i:05d}"
    for i in range(1, NUM_CUSTOMERS + 1)
]

SHUFFLE THE CUSTOMER POOL

In [6]:
np.random.seed(42)

shuffled_customers = np.random.permutation(customer_pool)

SPLIT CUSTOMERS INTO GROUPS 

In [7]:
regular = shuffled_customers[:1260]
frequent = shuffled_customers[1260:1620]
fleet = shuffled_customers[1620:]

In [8]:
weights = np.concatenate([
    np.ones(len(regular)) * 1,
    np.ones(len(frequent)) * 3,
    np.ones(len(fleet)) * 6
])

In [9]:
probabilities = weights / weights.sum()

In [10]:
print(probabilities.sum())

0.9999999999999998


In [11]:
assigned_customers = np.random.choice(
    shuffled_customers,
    size=NUM_SERVICE_VISITS,
    replace=True,
    p=probabilities
)

In [12]:
df = pd.DataFrame({
    "Job_card_ids": job_card_ids,
    "Customer_id" : assigned_customers

})

In [13]:
df.head()

,Job_card_ids,Customer_id
0,JC000001,C00256
1,JC000002,C00331
2,JC000003,C01076
3,JC000004,C00096
4,JC000005,C01486


In [14]:
NUM_VEHICLES  = 3091
vehicle_pool = [
    f"EV{i:05d}"
    for i in range(1,NUM_VEHICLES + 1)
]

In [15]:
print(vehicle_pool[:5])

['EV00001', 'EV00002', 'EV00003', 'EV00004', 'EV00005']


In [16]:
customer_vehicle_count = {}

In [17]:
def assign_vehicle_count(category):

    if category == "Regular":
        return np.random.choice(
            [1, 2],
            p=[0.90, 0.10]
        )

    elif category == "Frequent":
        return np.random.choice(
            [1, 2, 3],
            p=[0.30, 0.50, 0.20]
        )

    elif category == "Fleet":
        return np.random.randint(4, 9)

In [18]:
# Regular Customers
for customer in regular:
    customer_vehicle_count[customer] = assign_vehicle_count("Regular")

# Frequent Customers
for customer in frequent:
    customer_vehicle_count[customer] = assign_vehicle_count("Frequent")

# Fleet Customers
for customer in fleet:
    customer_vehicle_count[customer] = assign_vehicle_count("Fleet")

In [19]:
total_required_vehicles = sum(customer_vehicle_count.values())

print(total_required_vehicles)

3158


In [20]:
print(total_required_vehicles)
print(len(customer_vehicle_count))

3158
1800


In [21]:
NUM_VEHICLES = total_required_vehicles

vehicle_pool = [
    f"EV{i:05d}"
    for i in range(1, NUM_VEHICLES + 1)
]

In [22]:
customer_vehicle_mapping = []

vehicle_index = 0

In [23]:
for customer, num_vehicles in customer_vehicle_count.items():

    for _ in range(num_vehicles):

        customer_vehicle_mapping.append({
            "Customer_ID": customer,
            "Vehicle_ID": vehicle_pool[vehicle_index]
        })

        vehicle_index += 1

In [24]:
customer_vehicle_df = pd.DataFrame(customer_vehicle_mapping)

customer_vehicle_df.head()

,Customer_ID,Vehicle_ID
0,C01592,EV00001
1,C00944,EV00002
2,C00944,EV00003
3,C00870,EV00004
4,C00163,EV00005


In [25]:
customer_vehicle_df["Vehicle_ID"].duplicated().sum()

np.int64(0)

In [26]:
vehicle_models = [
    "Ather 450X",
    "Ather Rizta",
    "Ola S1 Pro",
    "TVS iQube",
    "Bajaj Chetak",
    "Hero Vida V1",
]

In [27]:
model_probabilities = [
    0.17, #Ather 450X
    0.18, # Ather Rizta
    0.12, # ola s1 Pro
    0.28, # TVS iQube
    0.20, # Bajaj Chetak
    0.05, # Hero Vida V1
]

In [28]:
vehicle_models_assigned = np.random.choice(
    vehicle_models,
    size=NUM_VEHICLES,
    p=model_probabilities
)

In [29]:
vehicle_master_df = pd.DataFrame({
    "Vehicle_id" : vehicle_pool,
    "Vehicle_Model" : vehicle_models_assigned
})

In [30]:
vehicle_master_df["Vehicle_Model"].value_counts(normalize=True)

Vehicle_Model
TVS iQube       0.291007
Bajaj Chetak    0.198543
Ather Rizta     0.184927
Ather 450X      0.159911
Ola S1 Pro      0.122863
Hero Vida V1    0.042749
Name: proportion, dtype: float64

In [31]:
current_year = 2026

model_launch_year = {
    "Ather 450X":   2020,
    "Ather Rizta" : 2024,
    "Ola S1 Pro" : 2021,
    "TVS iQube" : 2020,
    "Bajaj Chetak" : 2020,
    "Hero Vida V1" : 2022

}

In [32]:
manufacturing_year = []

for model in vehicle_master_df["Vehicle_Model"]:

    launch_year = model_launch_year[model]

    year = np.random.randint(
         launch_year,
         current_year + 1
    )

    manufacturing_year.append(year)

In [33]:
def generate_manufacturing_year(launch_year):

   # valid manufacturing years
   years = np.arange(launch_year,current_year + 1)

   # increasing weights 
   weights = np.arange(1,len(years) + 1)

   # convert weights to probabilities
   probabilities = weights/weights.sum()

   # generate manufacturing year
   return np.random.choice(
   years,
   p=probabilities
)



In [34]:
manufacturing_year = []

for model in vehicle_master_df["Vehicle_Model"]:
    launch_year = model_launch_year[model]

    year = generate_manufacturing_year(launch_year)

    manufacturing_year.append(year)

In [35]:
vehicle_master_df["Manufacturing_Year"] = manufacturing_year

In [36]:
vehicle_master_df.head()

,Vehicle_id,Vehicle_Model,Manufacturing_Year
0,EV00001,Hero Vida V1,2026
1,EV00002,TVS iQube,2022
2,EV00003,Bajaj Chetak,2025
3,EV00004,Ola S1 Pro,2026
4,EV00005,Ather Rizta,2026


In [37]:
vehicle_master_df["Vehicle_Age"] = (
    current_year - vehicle_master_df["Manufacturing_Year"]
)

In [38]:
vehicle_master_df.head()

,Vehicle_id,Vehicle_Model,Manufacturing_Year,Vehicle_Age
0,EV00001,Hero Vida V1,2026,0
1,EV00002,TVS iQube,2022,4
2,EV00003,Bajaj Chetak,2025,1
3,EV00004,Ola S1 Pro,2026,0
4,EV00005,Ather Rizta,2026,0


In [39]:
def generate_battery_age(vehicle_age):

    if vehicle_age == 0:
        return 0

    replacement_probability = battery_replacement_probability[vehicle_age]

    battery_replaced = np.random.choice(
        [True, False],
        p=[replacement_probability, 1 - replacement_probability]
    )

    if not battery_replaced:
        return vehicle_age

    battery_ages = np.arange(0, vehicle_age + 1)

    weights = np.arange(len(battery_ages), 0, -1)

    probabilities = weights / weights.sum()

    return np.random.choice(
        battery_ages,
        p=probabilities
    )

In [40]:
battery_replacement_probability = {
    0: 0.00,
    1: 0.02,
    2: 0.05,
    3: 0.10,
    4: 0.18,
    5: 0.30,
    6: 0.45
}

In [41]:
battery_age = []

for vehicle_age in vehicle_master_df["Vehicle_Age"]:
    battery_age.append(generate_battery_age(vehicle_age))

vehicle_master_df["Battery_age"] = battery_age


In [42]:
vehicle_master_df.columns

Index(['Vehicle_id', 'Vehicle_Model', 'Manufacturing_Year', 'Vehicle_Age',
       'Battery_age'],
      dtype='object')

In [43]:
# Validation 1
assert (
    vehicle_master_df["Battery_age"] <= vehicle_master_df["Vehicle_Age"]
).all(), "Battery age cannot exceed vehicle age."

In [44]:
# Validation 2
assert (
    vehicle_master_df["Battery_age"] >= 0
).all(), "Battery age cannot be negative."

In [45]:
# Validation 3
vehicle_master_df[["Vehicle_Age", "Battery_age"]].sample(10, random_state=42)

,Vehicle_Age,Battery_age
3053,3,3
2681,1,1
2961,2,2
2335,0,0
139,4,4
3075,0,0
2545,0,0
2384,0,0
1411,2,2
1421,0,0


In [46]:
battery_health_ranges = {
    0: (98, 100),
    1: (92, 98),
    2: (85, 94),
    3: (75, 88),
    4: (65, 80),
    5: (55, 72),
    6: (45, 65)
}

In [47]:
def generate_battery_health(battery_age):

    min_health,max_health = battery_health_ranges[battery_age]

    return np.random.randint(
         min_health,
         max_health + 1
    )

In [48]:
battery_health = []

for battery_age in vehicle_master_df["Battery_age"]:
    battery_health.append(
        generate_battery_health(battery_age)
    )

vehicle_master_df["Battery_Health"] = battery_health

In [49]:
vehicle_master_df[
    [
        "Vehicle_Age",
        "Battery_age",
        "Battery_Health"
    ]
].sample(10,random_state=42)

,Vehicle_Age,Battery_age,Battery_Health
3053,3,3,77
2681,1,1,94
2961,2,2,91
2335,0,0,98
139,4,4,69
3075,0,0,99
2545,0,0,99
2384,0,0,100
1411,2,2,86
1421,0,0,100


In [50]:
vehicle_master_df.groupby("Battery_age")["Battery_Health"].agg(
    ["min","max","mean","count"]
)

,min,max,mean,count
Battery_age,,,,
0,98,100,99.012000,1000
1,92,98,95.090802,848
2,85,94,89.517544,570
3,75,88,81.521858,366
4,65,80,72.309735,226
5,55,72,63.719626,107
6,46,65,56.487805,41


In [51]:
vehicle_master_df[["Battery_age","Battery_Health"]].describe()

,Battery_age,Battery_Health
count,3158.000000,3158.000000
mean,1.510766,90.559531
std,1.487050,10.418520
min,0.000000,46.000000
25%,0.000000,86.000000
50%,1.000000,94.000000
75%,2.000000,98.000000
max,6.000000,100.000000


# Service Records Dataset

Each row in this dataset represents one service visit of one vehicle.

A single vehicle can have multiple service visits over its lifetime.

This table will be used to generate operational features and the three ML targets:

- Repair Cost (Regression)
- Repair Time (Regression)
- Delay Risk (Classification)

In [52]:
SERVICE_VISIT_RANGES = {
    0: (1, 1),
    1: (1, 2),
    2: (2, 3),
    3: (3, 4),
    4: (4, 5),
    5: (5, 6),
    6: (6, 8)
}

In [53]:
def generate_number_of_service_visits(vehicle_age):

    min_visits,max_visits = SERVICE_VISIT_RANGES[vehicle_age]

    return np.random.randint(
        min_visits,
        max_visits + 1
    )


In [54]:
vehicle_master_df["Number_of_Service_Visits"] = (
    vehicle_master_df["Vehicle_Age"]
    .apply(generate_number_of_service_visits)
)

In [55]:
print(vehicle_master_df.columns.tolist())

['Vehicle_id', 'Vehicle_Model', 'Manufacturing_Year', 'Vehicle_Age', 'Battery_age', 'Battery_Health', 'Number_of_Service_Visits']


In [56]:
vehicle_master_df.rename(
    columns={"Vehicle_id": "Vehicle_ID"},
    inplace=True
)

In [57]:
vehicle_master_df[
    [
        "Vehicle_ID",
        "Vehicle_Age",
        "Number_of_Service_Visits"
    ]
].sample(10,random_state=42)

,Vehicle_ID,Vehicle_Age,Number_of_Service_Visits
3053,EV03054,3,4
2681,EV02682,1,1
2961,EV02962,2,2
2335,EV02336,0,1
139,EV00140,4,4
3075,EV03076,0,1
2545,EV02546,0,1
2384,EV02385,0,1
1411,EV01412,2,2
1421,EV01422,0,1


In [58]:
vehicle_master_df.groupby("Vehicle_Age")["Number_of_Service_Visits"].agg(
    ["min", "max", "mean", "count"]
)

,min,max,mean,count
Vehicle_Age,,,,
0,1,1,1.000000,917
1,1,2,1.493827,810
2,2,3,2.501779,562
3,3,4,3.514436,381
4,4,5,4.427536,276
5,5,6,5.540000,150
6,6,8,6.983871,62


In [59]:
vehicle_master_df["Number_of_Service_Visits"].value_counts().sort_index()

Number_of_Service_Visits
1    1327
2     680
3     467
4     354
5     187
6     102
7      21
8      20
Name: count, dtype: int64

In [60]:
vehicle_master_df[
    [
        "Vehicle_ID",
        "Vehicle_Age",
        "Number_of_Service_Visits"
    ]
].sample(15, random_state=42)

,Vehicle_ID,Vehicle_Age,Number_of_Service_Visits
3053,EV03054,3,4
2681,EV02682,1,1
2961,EV02962,2,2
2335,EV02336,0,1
139,EV00140,4,4
3075,EV03076,0,1
2545,EV02546,0,1
2384,EV02385,0,1
1411,EV01412,2,2
1421,EV01422,0,1


In [61]:
vehicle_master_df.groupby("Vehicle_Age")["Number_of_Service_Visits"].agg(
    ["min", "max", "mean", "count"]
)

,min,max,mean,count
Vehicle_Age,,,,
0,1,1,1.000000,917
1,1,2,1.493827,810
2,2,3,2.501779,562
3,3,4,3.514436,381
4,4,5,4.427536,276
5,5,6,5.540000,150
6,6,8,6.983871,62


In [62]:
vehicle_master_df[
    ["Vehicle_Age", "Number_of_Service_Visits"]
].sample(20, random_state=42)

,Vehicle_Age,Number_of_Service_Visits
3053,3,4
2681,1,1
2961,2,2
2335,0,1
139,4,4
3075,0,1
2545,0,1
2384,0,1
1411,2,2
1421,0,1


# Creating Service Records

Each vehicle can have multiple historical service visits.

In this step, we expand the Vehicle Master table into a Service Records table by creating one row for every service visit.

In [63]:
service_records = []
service_counter = 1

In [64]:
for _, vehicle in vehicle_master_df.iterrows():

    vehicle_id = vehicle["Vehicle_ID"]
    visits = vehicle["Number_of_Service_Visits"]

    for visit in range(visits):

        service_records.append(
            {
                "Service_ID": f"S{service_counter:06d}",
                "Vehicle_ID": vehicle_id
            }
        )

        service_counter += 1

In [65]:
service_records_df = pd.DataFrame(service_records)

In [66]:
service_records_df.head(10)

,Service_ID,Vehicle_ID
0,S000001,EV00001
1,S000002,EV00002
2,S000003,EV00002
3,S000004,EV00002
4,S000005,EV00002
5,S000006,EV00002
6,S000007,EV00003
7,S000008,EV00003
8,S000009,EV00004
9,S000010,EV00005


In [67]:
print(f"Number of Vehicles      : {len(vehicle_master_df):,}")
print(f"Number of Service Visits: {len(service_records_df):,}")

Number of Vehicles      : 3,158
Number of Service Visits: 7,358


In [68]:
service_records_df["Service_ID"].nunique()

7358

In [69]:
service_records_df["Vehicle_ID"].nunique()

3158

In [70]:
service_records_df.groupby("Vehicle_ID").size().head(10)

Vehicle_ID
EV00001    1
EV00002    5
EV00003    2
EV00004    1
EV00005    1
EV00006    2
EV00007    4
EV00008    1
EV00009    2
EV00010    1
dtype: int64

# Manufacturing Date

Generate a realistic manufacturing date for each vehicle.

This date acts as the starting point of the vehicle's lifecycle and will later be used to generate the purchase date and chronological service history.

In [71]:
from datetime import datetime, timedelta

In [72]:
def generate_manufacturing_date(manufacturing_year):
    start_date = datetime(manufacturing_year,1,1)
    end_date = datetime(manufacturing_year,12,31)

    random_days = np.random.randint(
        0,
        (end_date - start_date).days + 1
    )

    return start_date + timedelta(days=random_days)
    

In [73]:
vehicle_master_df["Manufacturing_Date"] = (
    vehicle_master_df["Manufacturing_Year"]
    .apply(generate_manufacturing_date)
)

In [74]:
vehicle_master_df[
    [
        "Manufacturing_Year",
        "Manufacturing_Date"
    ]
].sample(10,random_state=42)

,Manufacturing_Year,Manufacturing_Date
3053,2023,2023-07-14
2681,2025,2025-12-09
2961,2024,2024-09-19
2335,2026,2026-08-15
139,2022,2022-12-22
3075,2026,2026-11-16
2545,2026,2026-09-30
2384,2026,2026-06-18
1411,2024,2024-03-16
1421,2026,2026-05-17


# Purchase Date

Generate a realistic purchase date for each vehicle.

The purchase date is generated between 15 and 90 days after the manufacturing date.

It marks the beginning of customer ownership and will later be used to generate the service history and warranty period.

In [75]:
def generate_purchase_date(manufacturing_date):
    days_after_manufacturing = np.random.randint(15,91)
    return manufacturing_date + timedelta(days=days_after_manufacturing)

In [76]:
vehicle_master_df["Purchase_Date"] = (
    vehicle_master_df["Manufacturing_Date"]
    .apply(generate_purchase_date)
)

In [77]:
vehicle_master_df[
    [
        "Manufacturing_Date",
        "Purchase_Date"
    ]
].sample(10,random_state = 42)

,Manufacturing_Date,Purchase_Date
3053,2023-07-14,2023-08-27
2681,2025-12-09,2026-02-14
2961,2024-09-19,2024-10-22
2335,2026-08-15,2026-10-04
139,2022-12-22,2023-02-13
3075,2026-11-16,2027-02-04
2545,2026-09-30,2026-11-09
2384,2026-06-18,2026-08-01
1411,2024-03-16,2024-06-02
1421,2026-05-17,2026-07-11


In [78]:
(
    vehicle_master_df["Purchase_Date"] -
    vehicle_master_df["Manufacturing_Date"]
).dt.days.describe()

count    3158.000000
mean       52.258708
std        21.767362
min        15.000000
25%        34.000000
50%        52.000000
75%        71.000000
max        90.000000
dtype: float64

In [79]:
vehicle_master_df

,Vehicle_ID,Vehicle_Model,Manufacturing_Year,Vehicle_Age,Battery_age,Battery_Health,Number_of_Service_Visits,Manufacturing_Date,Purchase_Date
0,EV00001,Hero Vida V1,2026,0,0,100,1,2026-11-19,2026-12-22
1,EV00002,TVS iQube,2022,4,4,77,5,2022-06-16,2022-08-01
2,EV00003,Bajaj Chetak,2025,1,1,97,2,2025-03-30,2025-06-02
3,EV00004,Ola S1 Pro,2026,0,0,100,1,2026-11-12,2027-02-06
4,EV00005,Ather Rizta,2026,0,0,100,1,2026-01-11,2026-03-21
...,...,...,...,...,...,...,...,...,...
3153,EV03154,Bajaj Chetak,2026,0,0,98,1,2026-01-11,2026-03-11
3154,EV03155,Bajaj Chetak,2023,3,3,83,4,2023-05-19,2023-06-06
3155,EV03156,Ather Rizta,2026,0,0,100,1,2026-09-10,2026-10-10
3156,EV03157,Ather 450X,2021,5,5,70,6,2021-12-11,2022-03-01


In [80]:
DATASET_END_DATE = datetime(2026,12,31)

In [81]:
def generate_first_service_date(purchase_date):
    days_after_purchase = np.random.randint(90,241)
    return purchase_date + timedelta(days=days_after_purchase)

In [82]:
def generate_next_service_date(previous_service_date):

    days_after_previous = np.random.randint(120, 421)

    return previous_service_date + timedelta(days=days_after_previous)

In [83]:
print(vehicle_master_df.columns.tolist())

['Vehicle_ID', 'Vehicle_Model', 'Manufacturing_Year', 'Vehicle_Age', 'Battery_age', 'Battery_Health', 'Number_of_Service_Visits', 'Manufacturing_Date', 'Purchase_Date']


In [84]:
service_history = []

for _, vehicle in vehicle_master_df.iterrows():
    for visit_number in range(1,vehicle["Number_of_Service_Visits"] + 1):
        service_history.append({
            "Vehicle_ID": vehicle["Vehicle_ID"],
            "Visit_Number" : visit_number
        })
service_history_df = pd.DataFrame(service_history)


In [85]:
service_history_df = service_history_df.merge(
    vehicle_master_df[["Vehicle_ID","Purchase_Date"]],
    on="Vehicle_ID",
    how="left"
)


In [86]:
service_history_df.head(10)

,Vehicle_ID,Visit_Number,Purchase_Date
0,EV00001,1,2026-12-22
1,EV00002,1,2022-08-01
2,EV00002,2,2022-08-01
3,EV00002,3,2022-08-01
4,EV00002,4,2022-08-01
5,EV00002,5,2022-08-01
6,EV00003,1,2025-06-02
7,EV00003,2,2025-06-02
8,EV00004,1,2027-02-06
9,EV00005,1,2026-03-21


In [87]:
print(service_history_df.shape)

print(service_history_df["Purchase_Date"].isna().sum())

(7358, 3)
0


In [88]:
DATASET_CUTOFF_DATE = pd.Timestamp("2026-12-31")

In [89]:
service_dates = []

for vehicle_id, group in service_history_df.groupby("Vehicle_ID"):

    group = group.sort_values("Visit_Number")

    purchase_date = group["Purchase_Date"].iloc[0]

    previous_service_date = None

    for _, row in group.iterrows():

        if row["Visit_Number"] == 1:
            service_date = generate_first_service_date(purchase_date)

        else:
            service_date = generate_next_service_date(
                previous_service_date
            )

        service_dates.append({
            "Vehicle_ID": vehicle_id,
            "Visit_Number": row["Visit_Number"],
            "Service_Date": service_date
        })

        previous_service_date = service_date

In [90]:
service_dates_df = pd.DataFrame(service_dates)

service_dates_df.head(10)

,Vehicle_ID,Visit_Number,Service_Date
0,EV00001,1,2027-05-28
1,EV00002,1,2023-03-26
2,EV00002,2,2023-12-15
3,EV00002,3,2024-04-18
4,EV00002,4,2024-11-05
5,EV00002,5,2025-07-05
6,EV00003,1,2025-12-13
7,EV00003,2,2027-01-26
8,EV00004,1,2027-06-29
9,EV00005,1,2026-09-08


In [91]:
(service_dates_df["Service_Date"] > DATASET_CUTOFF_DATE).sum()

np.int64(775)

In [92]:
service_dates_df = service_dates_df[
    service_dates_df["Service_Date"] <= DATASET_CUTOFF_DATE
].copy()

In [93]:
print(len(service_dates_df))

print(
    (service_dates_df["Service_Date"] > DATASET_CUTOFF_DATE).sum()
)

6583
0


In [94]:
service_history_df = service_history_df.merge(
    service_dates_df,
    on=["Vehicle_ID","Visit_Number"],
    how = "inner"
)

In [95]:
print(service_history_df.shape)

service_history_df.head(10)

(6583, 4)


,Vehicle_ID,Visit_Number,Purchase_Date,Service_Date
0,EV00002,1,2022-08-01,2023-03-26
1,EV00002,2,2022-08-01,2023-12-15
2,EV00002,3,2022-08-01,2024-04-18
3,EV00002,4,2022-08-01,2024-11-05
4,EV00002,5,2022-08-01,2025-07-05
5,EV00003,1,2025-06-02,2025-12-13
6,EV00005,1,2026-03-21,2026-09-08
7,EV00006,1,2025-05-19,2025-11-18
8,EV00006,2,2025-05-19,2026-12-27
9,EV00007,1,2022-04-26,2022-09-03


In [96]:
(
    service_history_df["Service_Date"]
    < service_history_df["Purchase_Date"]
).sum()

np.int64(0)

In [97]:
(
    service_history_df["Service_Date"]
    > DATASET_CUTOFF_DATE
).sum()

np.int64(0)

In [98]:
service_history_df[
    service_history_df["Vehicle_ID"] == "EV00001"
]

,Vehicle_ID,Visit_Number,Purchase_Date,Service_Date


In [99]:
service_history_df = service_history_df.sort_values(
    ["Vehicle_ID", "Visit_Number"]
)

invalid_order = (
    service_history_df
    .groupby("Vehicle_ID")["Service_Date"]
    .diff()
    .dt.days
    .dropna()
    <= 0
).sum()

print(invalid_order)

0


In [100]:
service_history_df = service_history_df.merge(
    vehicle_master_df[
        [
            "Vehicle_ID",
            "Vehicle_Model",
            "Manufacturing_Date"
        ]
    ],
    on="Vehicle_ID",
    how="left"
)

In [101]:
service_history_df.head()

,Vehicle_ID,Visit_Number,Purchase_Date,Service_Date,Vehicle_Model,Manufacturing_Date
0,EV00002,1,2022-08-01,2023-03-26,TVS iQube,2022-06-16
1,EV00002,2,2022-08-01,2023-12-15,TVS iQube,2022-06-16
2,EV00002,3,2022-08-01,2024-04-18,TVS iQube,2022-06-16
3,EV00002,4,2022-08-01,2024-11-05,TVS iQube,2022-06-16
4,EV00002,5,2022-08-01,2025-07-05,TVS iQube,2022-06-16


In [102]:
print(service_history_df.shape)

print(
    service_history_df[
        ["Vehicle_Model", "Manufacturing_Date"]
    ].isnull().sum()
)

(6583, 6)
Vehicle_Model         0
Manufacturing_Date    0
dtype: int64


In [103]:
service_history_df["Vehicle_Age_at_Service"] = (
    (
        service_history_df["Service_Date"]
        - service_history_df["Manufacturing_Date"]
    ).dt.days / 365.25
).round(2)

In [104]:
service_history_df[
    [
        "Vehicle_ID",
        "Visit_Number",
        "Manufacturing_Date",
        "Service_Date",
        "Vehicle_Age_at_Service"
    ]
].head(10)

,Vehicle_ID,Visit_Number,Manufacturing_Date,Service_Date,Vehicle_Age_at_Service
0,EV00002,1,2022-06-16,2023-03-26,0.77
1,EV00002,2,2022-06-16,2023-12-15,1.50
2,EV00002,3,2022-06-16,2024-04-18,1.84
3,EV00002,4,2022-06-16,2024-11-05,2.39
4,EV00002,5,2022-06-16,2025-07-05,3.05
5,EV00003,1,2025-03-30,2025-12-13,0.71
6,EV00005,1,2026-01-11,2026-09-08,0.66
7,EV00006,1,2025-02-18,2025-11-18,0.75
8,EV00006,2,2025-02-18,2026-12-27,1.85
9,EV00007,1,2022-03-02,2022-09-03,0.51


In [105]:
(service_history_df["Vehicle_Age_at_Service"] < 0).sum()

np.int64(0)

In [106]:
service_history_df["Battery_Age_at_Service"] = (
    (
        service_history_df["Service_Date"]
        - service_history_df["Purchase_Date"]
    ).dt.days / 365.25
).round(2)

In [107]:
service_history_df[
    [
        "Vehicle_ID",
        "Visit_Number",
        "Purchase_Date",
        "Service_Date",
        "Battery_Age_at_Service"
    ]
].head(10)

,Vehicle_ID,Visit_Number,Purchase_Date,Service_Date,Battery_Age_at_Service
0,EV00002,1,2022-08-01,2023-03-26,0.65
1,EV00002,2,2022-08-01,2023-12-15,1.37
2,EV00002,3,2022-08-01,2024-04-18,1.71
3,EV00002,4,2022-08-01,2024-11-05,2.26
4,EV00002,5,2022-08-01,2025-07-05,2.93
5,EV00003,1,2025-06-02,2025-12-13,0.53
6,EV00005,1,2026-03-21,2026-09-08,0.47
7,EV00006,1,2025-05-19,2025-11-18,0.50
8,EV00006,2,2025-05-19,2026-12-27,1.61
9,EV00007,1,2022-04-26,2022-09-03,0.36


In [108]:
(service_history_df["Battery_Age_at_Service"] < 0).sum()

np.int64(0)

In [109]:
def generate_battery_health_at_service(battery_age):

    if battery_age < 1:
        low, high = 95, 100

    elif battery_age < 2:
        low, high = 90, 97

    elif battery_age < 3:
        low, high = 82, 92

    elif battery_age < 4:
        low, high = 72, 86

    elif battery_age < 5:
        low, high = 62, 78

    else:
        low, high = 50, 70

    return np.random.randint(low, high + 1)

In [110]:
def get_battery_replacement_probability(battery_health, battery_age):

    if battery_health >= 85:
        probability = 0.02

    elif battery_health >= 75:
        probability = 0.05

    elif battery_health >= 65:
        probability = 0.15

    elif battery_health >= 55:
        probability = 0.35

    else:
        probability = 0.65

    if battery_age >= 4:
        probability += 0.10

    elif battery_age >= 3:
        probability += 0.05

    return min(probability, 0.90)

In [111]:
battery_history = []

for vehicle_id, group in service_history_df.groupby("Vehicle_ID"):

    group = group.sort_values("Service_Date")

    battery_start_date = group["Purchase_Date"].iloc[0]

    degradation_rate = np.random.uniform(0.03, 0.08)

    for _, visit in group.iterrows():

        service_date = visit["Service_Date"]

        battery_age = (
            service_date - battery_start_date
        ).days / 365.25

        battery_health = 100 - (
            battery_age * degradation_rate * 100
        )

        battery_health = round(
            max(battery_health, 40),
            2
        )

        replacement_probability = get_battery_replacement_probability(
            battery_health,
            battery_age
        )

        battery_replaced = np.random.choice(
            [True, False],
            p=[
                replacement_probability,
                1 - replacement_probability
            ]
        )

        # MUST stay inside the visit loop
        battery_history.append({
            "Vehicle_ID": vehicle_id,
            "Visit_Number": visit["Visit_Number"],
            "Battery_Age_at_Service": round(battery_age, 2),
            "Battery_Health_at_Service": battery_health,
            "Battery_Replaced": battery_replaced
        })

        # MUST also stay inside the visit loop
        if battery_replaced:
            battery_start_date = service_date
            degradation_rate = np.random.uniform(0.03, 0.08)

In [112]:
battery_history_df = pd.DataFrame(battery_history)

In [113]:
battery_history_df.head(10)

,Vehicle_ID,Visit_Number,Battery_Age_at_Service,Battery_Health_at_Service,Battery_Replaced
0,EV00002,1,0.65,95.68,False
1,EV00002,2,1.37,90.86,False
2,EV00002,3,1.71,88.59,False
3,EV00002,4,2.26,84.92,False
4,EV00002,5,2.93,80.51,False
5,EV00003,1,0.53,96.82,True
6,EV00005,1,0.47,96.42,False
7,EV00006,1,0.50,97.40,False
8,EV00006,2,1.61,91.66,False
9,EV00007,1,0.36,98.56,False


In [114]:
print(len(service_history_df))
print(len(battery_history_df))

6583
6583


In [115]:
(battery_history_df["Battery_Age_at_Service"] < 0).sum()

np.int64(0)

In [116]:
battery_history_df["Battery_Health_at_Service"].describe()

count    6583.000000
mean       92.773594
std         6.085426
min        57.770000
25%        89.980000
50%        94.880000
75%        97.430000
max        99.240000
Name: Battery_Health_at_Service, dtype: float64

In [117]:
battery_history_df["Battery_Replaced"].value_counts()

Battery_Replaced
False    6393
True      190
Name: count, dtype: int64

In [118]:
replaced_vehicles = (
    battery_history_df[
        battery_history_df["Battery_Replaced"] == True
    ]["Vehicle_ID"]
    .unique()
)

len(replaced_vehicles)

182

In [119]:
vehicle = replaced_vehicles[0]

battery_history_df[
    battery_history_df["Vehicle_ID"] == vehicle
]

,Vehicle_ID,Visit_Number,Battery_Age_at_Service,Battery_Health_at_Service,Battery_Replaced
5,EV00003,1,0.53,96.82,True


In [120]:
service_history_df = service_history_df.merge(
    battery_history_df,
    on=["Vehicle_ID", "Visit_Number"],
    how="left"
)

In [121]:
service_history_df.shape

(6583, 11)

In [122]:
print(service_history_df.columns.tolist())

['Vehicle_ID', 'Visit_Number', 'Purchase_Date', 'Service_Date', 'Vehicle_Model', 'Manufacturing_Date', 'Vehicle_Age_at_Service', 'Battery_Age_at_Service_x', 'Battery_Age_at_Service_y', 'Battery_Health_at_Service', 'Battery_Replaced']


In [123]:
service_history_df.drop(
    columns=["Battery_Age_at_Service_x"],
    inplace=True
)

In [124]:
service_history_df.rename(
    columns={
        "Battery_Age_at_Service_y": "Battery_Age_at_Service"
    },
    inplace=True
)

In [125]:
print(service_history_df.columns.tolist())

['Vehicle_ID', 'Visit_Number', 'Purchase_Date', 'Service_Date', 'Vehicle_Model', 'Manufacturing_Date', 'Vehicle_Age_at_Service', 'Battery_Age_at_Service', 'Battery_Health_at_Service', 'Battery_Replaced']


In [126]:
service_history_df[
    [
        "Battery_Age_at_Service",
        "Battery_Health_at_Service",
        "Battery_Replaced"
    ]
].isnull().sum()

Battery_Age_at_Service       0
Battery_Health_at_Service    0
Battery_Replaced             0
dtype: int64

In [127]:
service_history_df.shape

(6583, 10)

In [128]:
def generate_issue_family(vehicle_age,battery_health):
    weights = base_issue_weights.copy()

    # battery condition effect
    if battery_health < 60:
        weights["Battery"] *= 3
    elif battery_health < 70:
        weights["Battery"] *= 2
    elif battery_health < 80:
        weights["Battery"] *= 1.5

    # vehicle age effect
    if vehicle_age >= 5:
        weights["Brake"] *= 1.5
        weights["Tyre"] *= 1.4
        weights["Suspension"] *= 1.6
        weights["Motor"] *= 1.4
        
    elif vehicle_age >= 3:
        weights["Brake"] *= 1.2
        weights["Tyre"] *= 1.2
        weights["Suspension"] *= 1.3
        weights["Motor"] *= 1.2

    total_weight = sum(weights.values())

    probabilities = [
        weight/total_weight
        for weight in weights.values()
    ]

    # select one issue family
    issue_family = np.random.choice(
        list(weights.keys()),
        p=probabilities
    )

    return issue_family
        

In [129]:
base_issue_weights = {
    "General Service": 30,
    "Brake": 20,
    "Tyre": 17,
    "Electrical": 12,
    "Battery": 10,
    "Suspension": 7,
    "Motor": 4
}

In [130]:
base_issue_weights

{'General Service': 30,
 'Brake': 20,
 'Tyre': 17,
 'Electrical': 12,
 'Battery': 10,
 'Suspension': 7,
 'Motor': 4}

In [131]:
service_history_df["Issue_Family"] = service_history_df.apply(
    lambda row: generate_issue_family(
        row["Vehicle_Age_at_Service"],
        row["Battery_Health_at_Service"]
    ),
    axis=1
)

In [132]:
service_history_df["Issue_Family"].value_counts()

Issue_Family
General Service    2016
Brake              1315
Tyre               1099
Electrical          766
Battery             657
Suspension          480
Motor               250
Name: count, dtype: int64

In [133]:
service_history_df["Issue_Family"].value_counts(
    normalize=True
).mul(100).round(2)

Issue_Family
General Service    30.62
Brake              19.98
Tyre               16.69
Electrical         11.64
Battery             9.98
Suspension          7.29
Motor               3.80
Name: proportion, dtype: float64

In [134]:
healthy = service_history_df[
    service_history_df["Battery_Health_at_Service"] >= 85
]

poor_health = service_history_df[
    service_history_df["Battery_Health_at_Service"] < 70
]

In [135]:
print(
    healthy["Issue_Family"]
    .value_counts(normalize=True)
)

print(
    poor_health["Issue_Family"]
    .value_counts(normalize=True)
)

Issue_Family
General Service    0.307798
Brake              0.198485
Tyre               0.165605
Electrical         0.117576
Battery            0.100706
Suspension         0.071613
Motor              0.038217
Name: proportion, dtype: float64
Issue_Family
General Service    0.333333
Battery            0.212121
Suspension         0.151515
Brake              0.121212
Electrical         0.090909
Tyre               0.060606
Motor              0.030303
Name: proportion, dtype: float64


In [136]:
print("Healthy:", len(healthy))
print("Poor Health:", len(poor_health))

Healthy: 5809
Poor Health: 33


In [137]:
service_history_df["Battery_Health_at_Service"].describe()

pd.cut(
    service_history_df["Battery_Health_at_Service"],
    bins=[0, 60, 70, 80, 85, 90, 95, 100],
    include_lowest=True
).value_counts().sort_index()

Battery_Health_at_Service
(-0.001, 60.0]       2
(60.0, 70.0]        31
(70.0, 80.0]       292
(80.0, 85.0]       450
(85.0, 90.0]       878
(90.0, 95.0]      1693
(95.0, 100.0]     3237
Name: count, dtype: int64

In [138]:
degradation_rate = np.random.uniform(0.04, 0.10)

In [139]:
exact_issues = {

    "General Service": [
        "Routine Maintenance",
        "Periodic Inspection",
        "Cleaning and Lubrication"
    ],

    "Brake": [
        "Brake Pad Wear",
        "Brake Disc Issue",
        "Brake Adjustment",
        "Brake Fluid Issue"
    ],

    "Tyre": [
        "Tyre Wear",
        "Puncture",
        "Wheel Alignment",
        "Low Tyre Pressure"
    ],

    "Electrical": [
        "Wiring Issue",
        "Sensor Failure",
        "Lighting Issue",
        "Switch Failure"
    ],

    "Battery": [
        "Battery Degradation",
        "Charging Failure",
        "Battery Overheating",
        "Battery Connection Issue"
    ],

    "Suspension": [
        "Shock Absorber Wear",
        "Front Fork Issue",
        "Suspension Noise",
        "Suspension Bush Wear"
    ],

    "Motor": [
        "Motor Overheating",
        "Motor Bearing Issue",
        "Motor Noise",
        "Motor Performance Issue"
    ]
}

In [140]:
exact_issue_weights = {

    "General Service": {
        "Routine Maintenance": 50,
        "Periodic Inspection": 30,
        "Cleaning and Lubrication": 20
    },

    "Brake": {
        "Brake Pad Wear": 45,
        "Brake Adjustment": 25,
        "Brake Disc Issue": 20,
        "Brake Fluid Issue": 10
    },

    "Tyre": {
        "Tyre Wear": 40,
        "Puncture": 30,
        "Low Tyre Pressure": 20,
        "Wheel Alignment": 10
    },

    "Electrical": {
        "Wiring Issue": 35,
        "Sensor Failure": 25,
        "Lighting Issue": 25,
        "Switch Failure": 15
    },

    "Battery": {
        "Battery Degradation": 40,
        "Charging Failure": 30,
        "Battery Connection Issue": 20,
        "Battery Overheating": 10
    },

    "Suspension": {
        "Shock Absorber Wear": 35,
        "Front Fork Issue": 30,
        "Suspension Noise": 20,
        "Suspension Bush Wear": 15
    },

    "Motor": {
        "Motor Bearing Issue": 35,
        "Motor Performance Issue": 30,
        "Motor Noise": 20,
        "Motor Overheating": 15
    }
}

In [141]:
def generate_exact_issue(issue_family):

    weights = exact_issue_weights[issue_family]

    issues = list(weights.keys())

    total_weight = sum(weights.values())

    probabilities = [
        weight / total_weight
        for weight in weights.values()
    ]

    return np.random.choice(
        issues,
        p=probabilities
    )

In [142]:
service_history_df["Exact_Issue"] = (
    service_history_df["Issue_Family"]
    .apply(generate_exact_issue)
)

In [143]:
service_history_df[
    ["Issue_Family", "Exact_Issue"]
].head(20)

,Issue_Family,Exact_Issue
0,Motor,Motor Performance Issue
1,Battery,Charging Failure
2,General Service,Cleaning and Lubrication
3,General Service,Periodic Inspection
4,General Service,Cleaning and Lubrication
5,Motor,Motor Noise
6,Suspension,Suspension Bush Wear
7,Tyre,Tyre Wear
8,Electrical,Lighting Issue
9,Electrical,Wiring Issue


In [144]:
invalid_issues = service_history_df.apply(
    lambda row:
        row["Exact_Issue"]
        not in exact_issue_weights[row["Issue_Family"]],
    axis=1
).sum()

print(invalid_issues)

0


In [145]:
def generate_parts_required(exact_issue, battery_replaced):

    # Hard consistency rule
    if battery_replaced:
        return True

    # Otherwise use issue-specific probability
    probability = parts_required_probability[exact_issue]

    return np.random.choice(
        [True, False],
        p=[probability, 1 - probability]
    )

In [146]:
service_history_df["Parts_Required"] = service_history_df.apply(
    lambda row: generate_parts_required(
        row["Exact_Issue"],
        row["Battery_Replaced"]
    ),
    axis=1
)

NameError: name 'parts_required_probability' is not defined

In [ ]:
service_history_df["Parts_Required"].value_counts()

In [ ]:
service_history_df["Parts_Required"].value_counts(
    normalize=True
).mul(100).round(2)

In [ ]:
invalid_battery_parts = service_history_df[
    (service_history_df["Battery_Replaced"] == True) &
    (service_history_df["Parts_Required"] == False)
]

print(len(invalid_battery_parts))

In [ ]:
parts_availability_probability = {

    # General Service
    "Routine Maintenance": 0.95,
    "Periodic Inspection": 0.95,
    "Cleaning and Lubrication": 0.95,

    # Brake
    "Brake Pad Wear": 0.90,
    "Brake Adjustment": 0.95,
    "Brake Disc Issue": 0.80,
    "Brake Fluid Issue": 0.90,

    # Tyre
    "Tyre Wear": 0.90,
    "Puncture": 0.95,
    "Low Tyre Pressure": 0.95,
    "Wheel Alignment": 0.95,

    # Electrical
    "Wiring Issue": 0.85,
    "Sensor Failure": 0.70,
    "Lighting Issue": 0.85,
    "Switch Failure": 0.80,

    # Battery
    "Battery Degradation": 0.65,
    "Charging Failure": 0.70,
    "Battery Connection Issue": 0.85,
    "Battery Overheating": 0.70,

    # Suspension
    "Shock Absorber Wear": 0.75,
    "Front Fork Issue": 0.70,
    "Suspension Noise": 0.85,
    "Suspension Bush Wear": 0.75,

    # Motor
    "Motor Bearing Issue": 0.60,
    "Motor Performance Issue": 0.65,
    "Motor Noise": 0.70,
    "Motor Overheating": 0.70
}

In [ ]:
def generate_parts_available(parts_required,exact_issue):

    # no parts required
    if not parts_required:
         return False

    probability = parts_availability_probability[exact_issue]

    return np.random.choice(
        [True,False],
        p=[probability,1 - probability]
    )

In [ ]:
service_history_df["Parts_Available"] = service_history_df.apply(
    lambda row: generate_parts_available(
        row["Parts_Required"],
        row["Exact_Issue"]
    ),
    axis=1
)

In [ ]:
pd.crosstab(
    service_history_df["Parts_Required"],
    service_history_df["Parts_Available"]
)

In [ ]:
invalid_parts = (
    (~service_history_df["Parts_Required"]) &
    (service_history_df["Parts_Available"])
).sum()

print(invalid_parts)

In [ ]:
def generate_part_ordered(parts_required,parts_available):

    if parts_required and not parts_available:
        return True

    return False

In [ ]:
service_history_df["Part_Ordered"] = service_history_df.apply(
    lambda row: generate_part_ordered(
        row["Parts_Required"],
        row["Parts_Available"]
    ),
    axis=1
)

In [ ]:
service_history_df["Part_Ordered"].value_counts()

In [ ]:
service_history_df["Part_Ordered"].value_counts(
    normalize=True
).mul(100).round(2)

In [ ]:
pd.crosstab(
    [
        service_history_df["Parts_Required"],
        service_history_df["Parts_Available"]
    ],
    service_history_df["Part_Ordered"]
)

In [ ]:
invalid_orders = (
    service_history_df["Part_Ordered"]
    != (
        service_history_df["Parts_Required"]
        & ~service_history_df["Parts_Available"]
    )
).sum()

print(invalid_orders)

In [ ]:
part_eta_ranges = {

    # General Service
    "Routine Maintenance": (1, 2),
    "Periodic Inspection": (1, 2),
    "Cleaning and Lubrication": (1, 2),

    # Brake
    "Brake Pad Wear": (1, 3),
    "Brake Adjustment": (1, 2),
    "Brake Disc Issue": (2, 5),
    "Brake Fluid Issue": (1, 3),

    # Tyre
    "Tyre Wear": (1, 3),
    "Puncture": (1, 2),
    "Low Tyre Pressure": (1, 2),
    "Wheel Alignment": (1, 2),

    # Electrical
    "Wiring Issue": (2, 4),
    "Sensor Failure": (3, 7),
    "Lighting Issue": (1, 4),
    "Switch Failure": (2, 5),

    # Battery
    "Battery Degradation": (4, 10),
    "Charging Failure": (3, 8),
    "Battery Connection Issue": (2, 5),
    "Battery Overheating": (3, 8),

    # Suspension
    "Shock Absorber Wear": (2, 6),
    "Front Fork Issue": (3, 7),
    "Suspension Noise": (2, 4),
    "Suspension Bush Wear": (2, 6),

    # Motor
    "Motor Bearing Issue": (4, 9),
    "Motor Performance Issue": (4, 10),
    "Motor Noise": (3, 8),
    "Motor Overheating": (3, 8)
}

In [ ]:
def generate_part_eta(part_ordered, exact_issue):

    if not part_ordered:
        return 0

    min_days, max_days = part_eta_ranges[exact_issue]

    return np.random.randint(
        min_days,
        max_days + 1
    )

In [ ]:
service_history_df["Part_ETA_Days"] = service_history_df.apply(
    lambda row: generate_part_eta(
        row["Part_Ordered"],
        row["Exact_Issue"]
    ),
    axis=1
)

In [ ]:
service_history_df["Part_ETA_Days"].describe()

In [ ]:
invalid_eta = (
    (
        (~service_history_df["Part_Ordered"]) &
        (service_history_df["Part_ETA_Days"] != 0)
    )
    |
    (
        (service_history_df["Part_Ordered"]) &
        (service_history_df["Part_ETA_Days"] <= 0)
    )
).sum()

print(invalid_eta)

In [ ]:
service_history_df.loc[
    service_history_df["Part_Ordered"],
    ["Exact_Issue", "Part_ETA_Days"]
].sample(10)

In [ ]:
workshop_capacity = 20

In [ ]:
def generate_active_jobs(service_date):

    day = service_date.dayofweek

    # Saturday / Sunday
    if day >= 5:
        active_jobs = np.random.randint(12, 25)

    # Monday - Friday
    else:
        active_jobs = np.random.randint(5, 21)

    return active_jobs

In [ ]:
service_history_df["Active_Jobs_On_Arrival"] = (
    service_history_df["Service_Date"]
    .apply(generate_active_jobs)
)

In [ ]:
service_history_df["Workshop_Capacity"] = workshop_capacity

In [ ]:
service_history_df["Workshop_Utilization"] = (
    service_history_df["Active_Jobs_On_Arrival"]
    / service_history_df["Workshop_Capacity"]
).round(2)

In [ ]:
service_history_df[
    [
        "Service_Date",
        "Active_Jobs_On_Arrival",
        "Workshop_Capacity",
        "Workshop_Utilization"
    ]
].head(10)

In [ ]:
service_history_df["Workshop_Utilization"].describe()

In [ ]:
service_history_df["Day_Type"] = (
    service_history_df["Service_Date"]
    .dt.dayofweek
    .apply(lambda x: "Weekend" if x >= 5 else "Weekday")
)

In [ ]:
service_history_df.groupby("Day_Type")[
    "Active_Jobs_On_Arrival"
].mean().round(2)

In [ ]:
service_history_df.groupby("Day_Type")[
    "Workshop_Utilization"
].mean().round(2)

In [ ]:
(service_history_df["Workshop_Utilization"] > 1).value_counts()

In [ ]:
(service_history_df["Workshop_Utilization"] > 1).value_counts(
    normalize=True
).mul(100).round(2)

In [ ]:
technicians = {
    "TECH_01": 1,
    "TECH_02": 2,
    "TECH_03": 2,
    "TECH_04": 3,
    "TECH_05": 3,
    "TECH_06": 4,
    "TECH_07": 5,
    "TECH_08": 6
}

In [ ]:
high_complexity_families = [
    "Battery",
    "Motor",
    "Electrical"
]

In [ ]:
def assign_technician(issue_family):

    technician_ids = list(technicians.keys())

    if issue_family in high_complexity_families:

        weights = np.array([
            technicians[tech] for tech in technician_ids
        ], dtype=float)

    else:
        weights = np.ones(len(technician_ids))

    probabilities = weights / weights.sum()

    return np.random.choice(
        technician_ids,
        p=probabilities
    )

In [ ]:
service_history_df["Technician_ID"] = (
    service_history_df["Issue_Family"]
    .apply(assign_technician)
)

In [ ]:
service_history_df["Technician_Experience_Years"] = (
    service_history_df["Technician_ID"]
    .map(technicians)
)

In [ ]:
service_history_df[
    [
        "Issue_Family",
        "Exact_Issue",
        "Technician_ID",
        "Technician_Experience_Years"
    ]
].head(15)

In [ ]:
service_history_df.groupby("Issue_Family")[
    "Technician_Experience_Years"
].mean().round(2)

In [ ]:
repair_complexity = {

    # General Service
    "Routine Maintenance": 1,
    "Periodic Inspection": 1,
    "Cleaning and Lubrication": 1,

    # Brake
    "Brake Pad Wear": 2,
    "Brake Adjustment": 1,
    "Brake Disc Issue": 2,
    "Brake Fluid Issue": 1,

    # Tyre
    "Tyre Wear": 2,
    "Puncture": 1,
    "Low Tyre Pressure": 1,
    "Wheel Alignment": 1,

    # Electrical
    "Wiring Issue": 2,
    "Sensor Failure": 3,
    "Lighting Issue": 1,
    "Switch Failure": 2,

    # Battery
    "Battery Degradation": 3,
    "Charging Failure": 3,
    "Battery Connection Issue": 2,
    "Battery Overheating": 3,

    # Suspension
    "Shock Absorber Wear": 2,
    "Front Fork Issue": 3,
    "Suspension Noise": 2,
    "Suspension Bush Wear": 2,

    # Motor
    "Motor Bearing Issue": 3,
    "Motor Performance Issue": 3,
    "Motor Noise": 2,
    "Motor Overheating": 3
}

In [ ]:
service_history_df["Repair_Complexity"] = (
    service_history_df["Exact_Issue"]
    .map(repair_complexity)
)

In [ ]:
service_history_df["Repair_Complexity"].isnull().sum()

In [ ]:
service_history_df["Repair_Complexity"].value_counts(
    normalize=True
).mul(100).round(2)

In [ ]:
service_history_df[
    [
        "Issue_Family",
        "Exact_Issue",
        "Repair_Complexity"
    ]
].sample(15, random_state=42)

In [ ]:
labor_hour_ranges = {
    1: (0.5, 1.5),
    2: (1.5, 3.5),
    3: (3.5, 7.0)
}

In [ ]:
def generate_base_labor_hours(complexity):

    min_hours, max_hours = labor_hour_ranges[complexity]

    return round(
        np.random.uniform(min_hours, max_hours),
        2
    )

In [ ]:
service_history_df["Base_Labor_Hours"] = (
    service_history_df["Repair_Complexity"]
    .apply(generate_base_labor_hours)
)

In [ ]:
service_history_df.groupby("Repair_Complexity")[
    "Base_Labor_Hours"
].agg(["count", "mean", "min", "max"]).round(2)

In [ ]:
service_history_df["Base_Labor_Hours"].describe()

In [ ]:
def get_technician_efficiency(experience):

    if experience >= 5:
        return 0.88

    elif experience >= 4:
        return 0.91

    elif experience >= 3:
        return 0.94

    elif experience >= 2:
        return 0.97

    else:
        return 1.00

In [ ]:
service_history_df["Technician_Efficiency"] = (
    service_history_df["Technician_Experience_Years"]
    .apply(get_technician_efficiency)
)

In [ ]:
service_history_df["Effective_Labor_Hours"] = (
    service_history_df["Base_Labor_Hours"]
    * service_history_df["Technician_Efficiency"]
).round(2)

In [ ]:
service_history_df[
    [
        "Exact_Issue",
        "Repair_Complexity",
        "Base_Labor_Hours",
        "Technician_Experience_Years",
        "Technician_Efficiency",
        "Effective_Labor_Hours"
    ]
].sample(10, random_state=42)

In [ ]:
(
    service_history_df["Effective_Labor_Hours"]
    > service_history_df["Base_Labor_Hours"]
).sum()

In [ ]:
def generate_workshop_wait_hours(utilization):

    if utilization <= 0.60:
        wait_hours = np.random.uniform(0, 0.5)

    elif utilization <= 0.80:
        wait_hours = np.random.uniform(0.5, 1.5)

    elif utilization <= 1.00:
        wait_hours = np.random.uniform(1.5, 3.0)

    else:
        wait_hours = np.random.uniform(3.0, 6.0)

    return round(wait_hours, 2)

In [ ]:
service_history_df["Workshop_Wait_Hours"] = (
    service_history_df["Workshop_Utilization"]
    .apply(generate_workshop_wait_hours)
)

In [ ]:
service_history_df.groupby(
    pd.cut(
        service_history_df["Workshop_Utilization"],
        bins=[0, 0.6, 0.8, 1.0, float("inf")]
    ),
    observed=False
)["Workshop_Wait_Hours"].mean().round(2)

In [ ]:
service_history_df[
    [
        "Active_Jobs_On_Arrival",
        "Workshop_Capacity",
        "Workshop_Utilization",
        "Workshop_Wait_Hours"
    ]
].sample(10, random_state=42)

In [ ]:
WORKSHOP_HOURS_PER_DAY = 8

In [ ]:
service_history_df["Operational_Time_Hours"] = (
    service_history_df["Effective_Labor_Hours"]
    + service_history_df["Workshop_Wait_Hours"]
).round(2)

In [ ]:
service_history_df["Operational_Time_Days"] = (
    service_history_df["Operational_Time_Hours"]
    / WORKSHOP_HOURS_PER_DAY
).round(2)

In [ ]:
service_history_df[
    [
        "Effective_Labor_Hours",
        "Workshop_Wait_Hours",
        "Operational_Time_Hours",
        "Operational_Time_Days"
    ]
].sample(10, random_state=42)

In [ ]:
service_history_df["Operational_Time_Days"].describe()

In [ ]:
def generate_extra_delay():

    return np.random.choice(
        [0, 0.25, 0.5, 1.0],
        p=[0.55, 0.25, 0.15, 0.05]
    )

In [ ]:
service_history_df["Extra_Operational_Delay_Days"] = [
    generate_extra_delay()
    for _ in range(len(service_history_df))
]

In [ ]:
service_history_df["Turnaround_Time_Days"] = (
    service_history_df["Operational_Time_Days"]
    + service_history_df["Part_ETA_Days"]
    + service_history_df["Extra_Operational_Delay_Days"]
).round(2)

In [ ]:
service_history_df["Turnaround_Time_Days"].describe()

In [ ]:
service_history_df.groupby("Part_Ordered")[
    "Turnaround_Time_Days"
].agg(["count", "mean", "median", "max"]).round(2)

In [ ]:
(
    service_history_df["Turnaround_Time_Days"]
    < service_history_df["Operational_Time_Days"]
).sum()

In [ ]:
parts_cost_ranges = {

    # General Service
    "Routine Maintenance": (100, 400),
    "Periodic Inspection": (0, 200),
    "Cleaning and Lubrication": (50, 250),

    # Brake
    "Brake Pad Wear": (400, 1200),
    "Brake Adjustment": (0, 200),
    "Brake Disc Issue": (1000, 2500),
    "Brake Fluid Issue": (200, 600),

    # Tyre
    "Tyre Wear": (1200, 3000),
    "Puncture": (100, 300),
    "Low Tyre Pressure": (0, 100),
    "Wheel Alignment": (100, 400),

    # Electrical
    "Wiring Issue": (300, 1200),
    "Sensor Failure": (800, 2500),
    "Lighting Issue": (300, 1200),
    "Switch Failure": (300, 1000),

    # Battery
    "Battery Degradation": (8000, 18000),
    "Charging Failure": (1500, 5000),
    "Battery Connection Issue": (200, 1000),
    "Battery Overheating": (1000, 4000),

    # Suspension
    "Shock Absorber Wear": (1500, 3500),
    "Front Fork Issue": (1200, 3000),
    "Suspension Noise": (300, 1000),
    "Suspension Bush Wear": (500, 1500),

    # Motor
    "Motor Bearing Issue": (1500, 4000),
    "Motor Performance Issue": (2000, 6000),
    "Motor Noise": (800, 2500),
    "Motor Overheating": (1500, 5000)
}

In [ ]:
def generate_parts_cost(exact_issue, parts_required):

    if not parts_required:
        return 0

    min_cost, max_cost = parts_cost_ranges[exact_issue]

    return np.random.randint(
        min_cost,
        max_cost + 1
    )

In [ ]:
service_history_df["Parts_Cost"] = service_history_df.apply(
    lambda row: generate_parts_cost(
        row["Exact_Issue"],
        row["Parts_Required"]
    ),
    axis=1
)

In [ ]:
service_history_df["Parts_Cost"].describe()

In [ ]:
invalid_parts_cost = (
    (~service_history_df["Parts_Required"])
    &
    (service_history_df["Parts_Cost"] != 0)
).sum()

print(invalid_parts_cost)

In [ ]:
service_history_df.groupby("Issue_Family")[
    "Parts_Cost"
].mean().round(2).sort_values(ascending=False)

In [ ]:
LABOR_RATE_PER_HOUR = 500

In [ ]:
service_history_df["Labor_Cost"] = (
    service_history_df["Base_Labor_Hours"]
    * LABOR_RATE_PER_HOUR
).round(0).astype(int)

In [ ]:
service_history_df["Labor_Cost"].describe()

In [ ]:
service_history_df.groupby("Repair_Complexity")[
    "Labor_Cost"
].agg(["mean", "min", "max"]).round(2)

In [ ]:
(service_history_df["Labor_Cost"] <= 0).sum()

In [ ]:
def generate_warranty_status(
    issue_family,
    vehicle_age,
    battery_age
):

    if issue_family == "Battery":
        return battery_age <= 3

    return vehicle_age <= 3

In [ ]:
service_history_df["Warranty_Status"] = service_history_df.apply(
    lambda row: generate_warranty_status(
        row["Issue_Family"],
        row["Vehicle_Age_at_Service"],
        row["Battery_Age_at_Service"]
    ),
    axis=1
)

In [ ]:
service_history_df["Warranty_Status"].value_counts(
    normalize=True
).mul(100).round(2)

In [ ]:
service_history_df.loc[
    service_history_df["Issue_Family"] == "Battery",
    ["Battery_Age_at_Service", "Warranty_Status"]
].sample(10, random_state=42)

In [ ]:
invalid_battery_warranty = (
    (service_history_df["Issue_Family"] == "Battery")
    &
    (service_history_df["Battery_Age_at_Service"] > 3)
    &
    (service_history_df["Warranty_Status"] == True)
).sum()

print(invalid_battery_warranty)

In [ ]:
non_warranty_issues = [
    "Routine Maintenance",
    "Periodic Inspection",
    "Cleaning and Lubrication",
    "Brake Pad Wear",
    "Brake Adjustment",
    "Brake Fluid Issue",
    "Tyre Wear",
    "Puncture",
    "Low Tyre Pressure",
    "Wheel Alignment"
]

In [ ]:
def generate_warranty_covered(warranty_status, exact_issue):

    # Vehicle/component already outside warranty
    if not warranty_status:
        return False

    # Wear, maintenance or external-damage items
    if exact_issue in non_warranty_issues:
        return False

    # Eligible failure while under warranty
    return True

In [ ]:
service_history_df["Warranty_Covered"] = service_history_df.apply(
    lambda row: generate_warranty_covered(
        row["Warranty_Status"],
        row["Exact_Issue"]
    ),
    axis=1
)

In [ ]:
pd.crosstab(
    service_history_df["Warranty_Status"],
    service_history_df["Warranty_Covered"]
)

In [ ]:
invalid_warranty = (
    (~service_history_df["Warranty_Status"])
    &
    (service_history_df["Warranty_Covered"])
).sum()

print(invalid_warranty)

In [ ]:
invalid_wear_warranty = (
    service_history_df["Exact_Issue"].isin(non_warranty_issues)
    &
    service_history_df["Warranty_Covered"]
).sum()

print(invalid_wear_warranty)

In [ ]:
def generate_misc_cost():

    return np.random.choice(
        [0, 100, 200, 300, 500],
        p=[0.35, 0.25, 0.20, 0.15, 0.05]
    )

In [ ]:
service_history_df["Misc_Cost"] = [
    generate_misc_cost()
    for _ in range(len(service_history_df))
]

In [ ]:
service_history_df["Repair_Cost"] = (
    service_history_df["Parts_Cost"]
    + service_history_df["Labor_Cost"]
    + service_history_df["Misc_Cost"]
)

In [ ]:
service_history_df["Repair_Cost"].describe()

In [ ]:
service_history_df.groupby("Issue_Family")[
    "Repair_Cost"
].agg(["count", "mean", "median", "max"]).round(2)

In [ ]:
(service_history_df["Repair_Cost"] <= 0).sum()

In [ ]:
expected_tat_by_complexity = {
    1: 1.0,
    2: 1.5,
    3: 2.5
}

In [ ]:
def generate_expected_tat(complexity, part_ordered, part_eta):

    expected_tat = expected_tat_by_complexity[complexity]

    if part_ordered:
        expected_tat += part_eta

    return expected_tat

In [ ]:
service_history_df["Expected_TAT_Days"] = service_history_df.apply(
    lambda row: generate_expected_tat(
        row["Repair_Complexity"],
        row["Part_Ordered"],
        row["Part_ETA_Days"]
    ),
    axis=1
)

In [ ]:
service_history_df["Expected_TAT_Days"].describe()

In [ ]:
service_history_df.groupby("Repair_Complexity")[
    "Expected_TAT_Days"
].mean().round(2)

In [ ]:
service_history_df.groupby("Part_Ordered")[
    "Expected_TAT_Days"
].mean().round(2)

In [ ]:
service_history_df["Is_Delayed"] = (
    service_history_df["Turnaround_Time_Days"]
    > service_history_df["Expected_TAT_Days"]
)

In [ ]:
service_history_df["Is_Delayed"].value_counts()

In [ ]:
service_history_df["Is_Delayed"].value_counts(
    normalize=True
).mul(100).round(2)

In [ ]:
service_history_df.rename(
    columns={
        "Part_ETA_Days": "Expected_Part_ETA_Days"
    },
    inplace=True
)

In [ ]:
def generate_actual_part_wait(part_ordered, expected_eta):

    if not part_ordered:
        return 0

    delay = np.random.choice(
        [-1, 0, 1, 2, 3],
        p=[0.05, 0.65, 0.15, 0.10, 0.05]
    )

    actual_wait = expected_eta + delay

    return max(1, actual_wait)

In [ ]:
service_history_df["Actual_Part_Wait_Days"] = service_history_df.apply(
    lambda row: generate_actual_part_wait(
        row["Part_Ordered"],
        row["Expected_Part_ETA_Days"]
    ),
    axis=1
)

In [ ]:
service_history_df[
    [
        "Part_Ordered",
        "Expected_Part_ETA_Days",
        "Actual_Part_Wait_Days"
    ]
].sample(15, random_state=42)

In [ ]:
service_history_df.loc[
    service_history_df["Part_Ordered"],
    ["Expected_Part_ETA_Days", "Actual_Part_Wait_Days"]
].describe()

In [ ]:
invalid_actual_wait = (
    (~service_history_df["Part_Ordered"])
    &
    (service_history_df["Actual_Part_Wait_Days"] != 0)
).sum()

print(invalid_actual_wait)

In [ ]:
service_history_df.loc[
    service_history_df["Part_Ordered"],
    "Actual_Part_Wait_Days"
].sub(
    service_history_df.loc[
        service_history_df["Part_Ordered"],
        "Expected_Part_ETA_Days"
    ]
).value_counts().sort_index()

In [ ]:
def generate_expected_tat(
    complexity,
    part_ordered,
    expected_part_eta
):

    expected_tat = expected_tat_by_complexity[complexity]

    if part_ordered:
        expected_tat += expected_part_eta

    return expected_tat

In [ ]:
service_history_df["Expected_TAT_Days"] = service_history_df.apply(
    lambda row: generate_expected_tat(
        row["Repair_Complexity"],
        row["Part_Ordered"],
        row["Expected_Part_ETA_Days"]
    ),
    axis=1
)

In [ ]:
service_history_df["Is_Delayed"] = (
    service_history_df["Turnaround_Time_Days"]
    > service_history_df["Expected_TAT_Days"]
)

In [ ]:
service_history_df["Is_Delayed"].value_counts(
    normalize=True
).mul(100).round(2)

In [ ]:
service_history_df["Supplier_Delay_Days"] = (
    service_history_df["Actual_Part_Wait_Days"]
    - service_history_df["Expected_Part_ETA_Days"]
)

In [ ]:
service_history_df.groupby(
    service_history_df["Supplier_Delay_Days"] > 0
)["Is_Delayed"].mean().mul(100).round(2)

In [ ]:
service_history_df["Is_Delayed"].value_counts(
    normalize=True
).mul(100).round(2)

In [ ]:
service_history_df["Is_Delayed"].value_counts()

In [ ]:
def generate_extra_delay(workshop_utilization):

    # Normal workshop load
    if workshop_utilization <= 0.80:
        return np.random.choice(
            [0, 0.25, 0.5, 1.0],
            p=[0.65, 0.20, 0.10, 0.05]
        )

    # Busy workshop
    elif workshop_utilization <= 1.00:
        return np.random.choice(
            [0, 0.25, 0.5, 1.0, 1.5],
            p=[0.40, 0.20, 0.20, 0.15, 0.05]
        )

    # Overloaded workshop
    else:
        return np.random.choice(
            [0, 0.5, 1.0, 1.5, 2.0],
            p=[0.20, 0.20, 0.30, 0.20, 0.10]
        )

In [ ]:
service_history_df["Extra_Operational_Delay_Days"] = (
    service_history_df["Workshop_Utilization"]
    .apply(generate_extra_delay)
)

In [ ]:
service_history_df["Turnaround_Time_Days"] = (
    service_history_df["Operational_Time_Days"]
    + service_history_df["Actual_Part_Wait_Days"]
    + service_history_df["Extra_Operational_Delay_Days"]
).round(2)

In [ ]:
service_history_df["Is_Delayed"] = (
    service_history_df["Turnaround_Time_Days"]
    > service_history_df["Expected_TAT_Days"]
)

In [ ]:
service_history_df["Is_Delayed"].value_counts(
    normalize=True
).mul(100).round(2)

In [ ]:
service_history_df.groupby(
    service_history_df["Workshop_Utilization"] > 1
)["Is_Delayed"].mean().mul(100).round(2)

In [ ]:
print(service_history_df.shape)

print(service_history_df.columns.tolist())

In [ ]:
service_history_df.head()

In [ ]:
service_history_df["Workshop_Capacity"].value_counts()

In [ ]:
service_history_df.nunique().sort_values()

In [ ]:
service_history_df.isnull().sum().sort_values(ascending=False)

In [ ]:
service_history_df.duplicated(
    subset=["Vehicle_ID", "Visit_Number"]
).sum()

In [ ]:
invalid_service_dates = (
    service_history_df["Service_Date"]
    < service_history_df["Purchase_Date"]
).sum()

print("Service before purchase:", invalid_service_dates)

In [ ]:
print(
    "Negative vehicle age:",
    (service_history_df["Vehicle_Age_at_Service"] < 0).sum()
)

print(
    "Negative battery age:",
    (service_history_df["Battery_Age_at_Service"] < 0).sum()
)

print(
    "Invalid battery health:",
    (
        (service_history_df["Battery_Health_at_Service"] < 0)
        |
        (service_history_df["Battery_Health_at_Service"] > 100)
    ).sum()
)

In [ ]:
print(
    "Invalid repair cost:",
    (service_history_df["Repair_Cost"] <= 0).sum()
)

print(
    "Invalid TAT:",
    (service_history_df["Turnaround_Time_Days"] <= 0).sum()
)

In [ ]:
# 1. Battery replaced → parts must be required
check_1 = (
    service_history_df["Battery_Replaced"]
    & ~service_history_df["Parts_Required"]
).sum()


# 2. Parts not required → cannot be available
check_2 = (
    ~service_history_df["Parts_Required"]
    & service_history_df["Parts_Available"]
).sum()


# 3. Part ordered only when required AND unavailable
check_3 = (
    service_history_df["Part_Ordered"]
    != (
        service_history_df["Parts_Required"]
        & ~service_history_df["Parts_Available"]
    )
).sum()


# 4. No part ordered → expected ETA must be 0
check_4 = (
    ~service_history_df["Part_Ordered"]
    & (service_history_df["Expected_Part_ETA_Days"] != 0)
).sum()


# 5. No part ordered → actual wait must be 0
check_5 = (
    ~service_history_df["Part_Ordered"]
    & (service_history_df["Actual_Part_Wait_Days"] != 0)
).sum()


# 6. Warranty covered → must actually be under warranty
check_6 = (
    service_history_df["Warranty_Covered"]
    & ~service_history_df["Warranty_Status"]
).sum()


# 7. Effective labor should not exceed base labor
check_7 = (
    service_history_df["Effective_Labor_Hours"]
    > service_history_df["Base_Labor_Hours"]
).sum()


# 8. Repair cost must equal its generated components
check_8 = (
    service_history_df["Repair_Cost"]
    != (
        service_history_df["Parts_Cost"]
        + service_history_df["Labor_Cost"]
        + service_history_df["Misc_Cost"]
    )
).sum()


print("Battery/parts inconsistency:", check_1)
print("Parts availability inconsistency:", check_2)
print("Part ordering inconsistency:", check_3)
print("Expected ETA inconsistency:", check_4)
print("Actual wait inconsistency:", check_5)
print("Warranty inconsistency:", check_6)
print("Labor inconsistency:", check_7)
print("Repair cost inconsistency:", check_8)

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
service_history_df.to_csv(
    DATA_DIR / "ev_service_history_raw.csv",
    index=False
)

In [ ]:
print(service_history_df.shape)

print(
    "Saved:",
    DATA_DIR / "ev_service_history_raw.csv"
)

In [ ]:
# Identification / tracking columns
id_columns = [
    "Vehicle_ID",
    "Visit_Number",
    "Purchase_Date",
    "Service_Date",
    "Manufacturing_Date"
]


# Prediction-time candidate features
feature_columns = [
    "Vehicle_Model",
    "Vehicle_Age_at_Service",
    "Battery_Age_at_Service",
    "Battery_Health_at_Service",

    "Issue_Family",
    "Exact_Issue",

    "Parts_Required",
    "Parts_Available",
    "Part_Ordered",
    "Expected_Part_ETA_Days",

    "Active_Jobs_On_Arrival",
    "Workshop_Utilization",
    "Day_Type",

    "Technician_ID",
    "Technician_Experience_Years",

    "Repair_Complexity",

    "Warranty_Status",
    "Warranty_Covered"
]


# ML targets
target_columns = [
    "Repair_Cost",
    "Turnaround_Time_Days",
    "Is_Delayed"
]


# Simulation/helper/outcome columns
helper_columns = [
    "Battery_Replaced",

    "Workshop_Capacity",

    "Base_Labor_Hours",
    "Technician_Efficiency",
    "Effective_Labor_Hours",

    "Workshop_Wait_Hours",
    "Operational_Time_Hours",
    "Operational_Time_Days",

    "Extra_Operational_Delay_Days",

    "Parts_Cost",
    "Labor_Cost",
    "Misc_Cost",

    "Expected_TAT_Days",

    "Actual_Part_Wait_Days",
    "Supplier_Delay_Days"
]

In [ ]:
classified_columns = (
    id_columns
    + feature_columns
    + target_columns
    + helper_columns
)

print("Dataset columns:", len(service_history_df.columns))
print("Classified columns:", len(classified_columns))

In [ ]:
unclassified = set(service_history_df.columns) - set(classified_columns)

duplicates = [
    col for col in set(classified_columns)
    if classified_columns.count(col) > 1
]

print("Unclassified:", unclassified)
print("Duplicated:", duplicates)

In [ ]:
X = service_history_df.drop(columns=["Repair_Cost"])

In [ ]:
del X

In [148]:
service_history_df

,Vehicle_ID,Visit_Number,Purchase_Date,Service_Date,Vehicle_Model,Manufacturing_Date,Vehicle_Age_at_Service,Battery_Age_at_Service,Battery_Health_at_Service,Battery_Replaced,Issue_Family,Exact_Issue
0,EV00002,1,2022-08-01,2023-03-26,TVS iQube,2022-06-16,0.77,0.65,95.68,False,Motor,Motor Performance Issue
1,EV00002,2,2022-08-01,2023-12-15,TVS iQube,2022-06-16,1.50,1.37,90.86,False,Battery,Charging Failure
2,EV00002,3,2022-08-01,2024-04-18,TVS iQube,2022-06-16,1.84,1.71,88.59,False,General Service,Cleaning and Lubrication
3,EV00002,4,2022-08-01,2024-11-05,TVS iQube,2022-06-16,2.39,2.26,84.92,False,General Service,Periodic Inspection
4,EV00002,5,2022-08-01,2025-07-05,TVS iQube,2022-06-16,3.05,2.93,80.51,False,General Service,Cleaning and Lubrication
...,...,...,...,...,...,...,...,...,...,...,...,...
6578,EV03157,3,2022-03-01,2024-02-19,Ather 450X,2021-12-11,2.19,1.97,93.94,False,Brake,Brake Disc Issue
6579,EV03157,4,2022-03-01,2025-03-04,Ather 450X,2021-12-11,3.23,3.01,90.74,False,Tyre,Puncture
6580,EV03157,5,2022-03-01,2025-12-06,Ather 450X,2021-12-11,3.99,3.77,88.41,False,General Service,Routine Maintenance
6581,EV03157,6,2022-03-01,2026-05-31,Ather 450X,2021-12-11,4.47,4.25,86.93,False,Electrical,Switch Failure
